In [ ]:
import os
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from math import atan2, degrees

In [ ]:
import os
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from math import atan2, degrees

# ============ CONFIG: carpetas y CSV de salida ============
configs = [
    # (nombre_logico, carpeta_imagenes, nombre_csv_salida)
    ("real",  r"PROJECT_ROOT\OneDrive\Desktop\capstone\personaFeed_Juliana",      "landmarks_user_a.csv"),
    ("fake1", r"PROJECT_ROOT\OneDrive\Desktop\capstone\Screenshots user_b",      "landmarks_user_b.csv"),
    ("fake2", r"PROJECT_ROOT\OneDrive\Desktop\capstone\Screenshots user_c",         "landmarks_user_c.csv"),
]

# límite de frames por usuario
MAX_FRAMES = 170

# MediaPipe setup
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True)

# distance between points
def dist(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# angle
def angle(a, b, c):
    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba)*np.linalg.norm(bc))
    return np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))

def eye_aspect_ratio(pts):
    A = dist(pts[1], pts[5])
    B = dist(pts[2], pts[4])
    C = dist(pts[0], pts[3])
    return (A + B) / (2.0 * C) if C != 0 else 0

def mouth_aspect_ratio(top_lip, bottom_lip, left_mouth, right_mouth):
    vertical = dist(top_lip, bottom_lip)
    horizontal = dist(left_mouth, right_mouth)
    return vertical / horizontal if horizontal != 0 else 0


# ============ LOOP: procesar cada carpeta ============
for user_name, IMAGE_FOLDER, OUTPUT_CSV in configs:
    print(f"\nProcesando usuario: {user_name}")
    print(f"Carpeta: {IMAGE_FOLDER}")
    data_rows = []  # muy importante resetear aquí

    if not os.path.isdir(IMAGE_FOLDER):
        print(f"⚠ La carpeta no existe: {IMAGE_FOLDER}")
        continue

    # lista de archivos limitada a MAX_FRAMES
    all_files = sorted(os.listdir(IMAGE_FOLDER))
    image_files = [f for f in all_files if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    image_files = image_files[:MAX_FRAMES]  # 👈 aquí se limita a 170

    print(f"Se procesarán hasta {len(image_files)} imágenes para {user_name}")

    for filename in image_files:
        img_path = os.path.join(IMAGE_FOLDER, filename)
        image = cv2.imread(img_path)
        if image is None:
            print(f"Error loading {img_path}")
            continue
        
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image_rgb)

        if not results.multi_face_landmarks:
            print(f"No face detected in {filename}")
            continue

        # extract the landmarks
        lm = results.multi_face_landmarks[0].landmark
        h, w, _ = image.shape

        # Cordinates convertions
        pts = [(lm[i].x * w, lm[i].y * h, lm[i].z) for i in range(len(lm))]

        # Landmarks 
        RIGHT_EYE = [33, 160, 158, 133, 153, 144]   # EAR right eye
        LEFT_EYE  = [362, 385, 387, 263, 373, 380]  # EAR left eye

        MOUTH_TOP    = 13
        MOUTH_BOTTOM = 14
        MOUTH_LEFT   = 78
        MOUTH_RIGHT  = 308

        # Brow inner distance
        BROW_LEFT  = 70
        BROW_RIGHT = 300

        # --- FEATURES ---
        fe = {}

        # EAR 
        fe["ear_left"]  = eye_aspect_ratio([pts[i] for i in LEFT_EYE])
        fe["ear_right"] = eye_aspect_ratio([pts[i] for i in RIGHT_EYE])

        # MAR (mouth movement)
        fe["mar"] = mouth_aspect_ratio(
            pts[MOUTH_TOP], pts[MOUTH_BOTTOM], pts[MOUTH_LEFT], pts[MOUTH_RIGHT]
        )

        # Distance between eyebrows 
        fe["brow_distance"] = dist(pts[BROW_LEFT], pts[BROW_RIGHT])

        # blinking
        fe["eye_open_left"]  = dist(pts[385], pts[387])
        fe["eye_open_right"] = dist(pts[160], pts[158])

        # mouth opening
        fe["mouth_vertical"]   = dist(pts[MOUTH_TOP], pts[MOUTH_BOTTOM])
        fe["mouth_horizontal"] = dist(pts[MOUTH_LEFT], pts[MOUTH_RIGHT])

        # --- head movement (pitch, yaw, roll) ---
        nose_tip   = np.array(pts[1])
        chin       = np.array(pts[152])
        left_cheek = np.array(pts[234])
        right_cheek= np.array(pts[454])

        # Pitch
        fe["pitch"] = degrees(atan2(chin[1] - nose_tip[1], chin[2] - nose_tip[2]))
        # Yaw
        fe["yaw"]   = degrees(atan2(right_cheek[0] - left_cheek[0], right_cheek[2] - left_cheek[2]))
        # Roll
        fe["roll"]  = degrees(atan2(right_cheek[1] - left_cheek[1], right_cheek[0] - left_cheek[0]))

        # 468 landmarks + info de usuario
        row = {"filename": filename, "user": user_name}
        for i in range(len(pts)):
            row[f"x_{i}"] = pts[i][0]
            row[f"y_{i}"] = pts[i][1]
            row[f"z_{i}"] = pts[i][2]

        row.update(fe)
        data_rows.append(row)

    #  CSV para este usuario
    df = pd.DataFrame(data_rows)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Archivo generado: {OUTPUT_CSV} con {len(df)} filas")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# === Lista de usuarios y sus CSV ===
files = [
    ("real",  "landmarks_user_a.csv"),
    ("fake1", "landmarks_user_b.csv"),
    ("fake2", "landmarks_user_c.csv"),
]

for label, csv_path in files:
    print(f"\n=== Usuario: {label} | CSV: {csv_path} ===")
    df = pd.read_csv(csv_path)
    print("Filas en este CSV:", len(df))

    if df.empty:
        print("⚠ CSV vacío, me lo salto.")
        continue

    # -----------------------------
    # 1) Elegir qué snapshot usar
    #    (fila 25 o la última si hay menos)
    # -----------------------------
    idx = min(22, len(df) - 1)
    row = df.iloc[idx]

    # Si tienes columna filename, la mostramos:
    if "filename" in df.columns:
        print(f"Usando fila idx = {idx}, filename = {row['filename']}")
    else:
        print(f"Usando fila idx = {idx} (no hay columna filename)")

    # -----------------------------
    # 2) Extraer coordenadas originales
    # -----------------------------
    xs = np.array([row[f"x_{i}"] for i in range(468)])
    ys = np.array([row[f"y_{i}"] for i in range(468)])
    zs = np.array([row[f"z_{i}"] for i in range(468)])

    # Centrar y escalar
    X = xs - xs.mean()
    Y = ys - ys.mean()
    Z = (zs - zs.mean()) * 200  # para ver mejor la profundidad

    # -----------------------------
    # 3) FRONTAL (sin rotar)
    # -----------------------------
    X_front = X
    Y_front = Y
    Z_front = Z

    # -----------------------------
    # 4) PERFIL DERECHO (rotar 90° alrededor de Y)
    #    Si quieres que mire al otro lado, cambia theta a -90
    # -----------------------------
    theta = np.radians(90)   # 90° → lado derecho (o izquierdo según ejes)
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    X_prof = X * cos_t + Z * sin_t
    Y_prof = Y
    Z_prof = -X * sin_t + Z * cos_t

    # -----------------------------
    # 5) Dibujar ambas vistas con la MISMA cámara (90, 90)
    # -----------------------------
    fig = plt.figure(figsize=(12, 5))

    # FRONTAL
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax1.scatter(X_front, Y_front, Z_front, s=5)
    ax1.set_title(f"3D Mesh - {label} (frontal)")
    ax1.set_xlabel("X")
    ax1.set_ylabel("Y")
    ax1.set_zlabel("Z")
    ax1.view_init(elev=90, azim=90)  # la vista que a ti te gustó

    # PERFIL
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    ax2.scatter(X_prof, Y_prof, Z_prof, s=5)
    ax2.set_title(f"3D Mesh - {label} (right profile)")
    ax2.set_xlabel("X'")
    ax2.set_ylabel("Y'")
    ax2.set_zlabel("Z'")
    ax2.view_init(elev=90, azim=90)  # misma cámara, puntos rotados

    plt.tight_layout()
    plt.show()


In [ ]:
import os
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt

# ==========================
# CONFIG: carpetas por usuario
# ==========================
configs = [
    ("real",
     r"PROJECT_ROOT\OneDrive\Desktop\capstone\personaFeed_Juliana"),
    ("fake1",
     r"PROJECT_ROOT\OneDrive\Desktop\capstone\Screenshots user_b"),
    ("fake2",
     r"PROJECT_ROOT\OneDrive\Desktop\capstone\Screenshots user_c"),
]

mp_face_mesh = mp.solutions.face_mesh

# Dibujamos en BLANCO, finito
drawing_spec_points = mp.solutions.drawing_utils.DrawingSpec(
    color=(255, 255, 255), thickness=1, circle_radius=1
)
drawing_spec_connections = mp.solutions.drawing_utils.DrawingSpec(
    color=(255, 255, 255), thickness=1, circle_radius=1
)

face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, refine_landmarks=True)

for label, IMAGE_FOLDER in configs:
    print(f"\n=== Usuario: {label} ===")
    print("Carpeta:", IMAGE_FOLDER)

    if not os.path.isdir(IMAGE_FOLDER):
        print("⚠ La carpeta no existe, lo salto.")
        continue

    # Tomamos todas las imágenes válidas
    images = sorted(
        [f for f in os.listdir(IMAGE_FOLDER)
         if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    )

    if not images:
        print("⚠ No se encontraron imágenes en esta carpeta.")
        continue

    # Elegimos un frame “representativo”: idx 25 o el último
    idx = 30 if len(images) > 22 else len(images) - 1
    img_name = images[idx]
    img_path = os.path.join(IMAGE_FOLDER, img_name)

    print(f"Usando imagen idx={idx}: {img_name}")

    # Leer imagen
    image = cv2.imread(img_path)
    if image is None:
        print("⚠ Error leyendo la imagen, lo salto.")
        continue

    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Pasar por MediaPipe FaceMesh
    results = face_mesh.process(rgb)

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            mp.solutions.drawing_utils.draw_landmarks(
                image=rgb,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=drawing_spec_points,
                connection_drawing_spec=drawing_spec_connections,
            )
    else:
        print("⚠ No se detectó cara en esta imagen.")
        continue

    # Mostrar overlay
    plt.figure(figsize=(6, 6))
    plt.imshow(rgb)
    plt.title(f"{label} - mesh overlay ({img_name})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Para que los gráficos se vean dentro del notebook
%matplotlib inline

# --- Cargar CSVs ---
df_real   = pd.read_csv("landmarks_user_a.csv")
df_fake1  = pd.read_csv("landmarks_user_b.csv")
df_fake2  = pd.read_csv("landmarks_user_c.csv")

# Añadimos columna label explícita (por si acaso)
df_real["label"]  = "real"
df_fake1["label"] = "fake1"
df_fake2["label"] = "fake2"

# Unimos todo
df_all = pd.concat([df_real, df_fake1, df_fake2], ignore_index=True)

print("Shape total:", df_all.shape)
print("\nFilas por label:")
print(df_all["label"].value_counts())
print("\nColumnas principales:")
print(df_all.columns[:20])


In [ ]:
feature_cols = [
    "ear_left", "ear_right",
    "mar", "mouth_vertical", "mouth_horizontal",
    "brow_distance",
    "eye_open_left", "eye_open_right",
    "pitch", "yaw", "roll"
]

# Por si acaso, nos quedamos solo con las que existan
feature_cols = [c for c in feature_cols if c in df_all.columns]
print("Features usadas:", feature_cols)

# DataFrame reducido para EDA
df_feat = df_all[["filename", "label"] + feature_cols].copy()

print("\nNaNs por columna:")
print(df_feat.isna().sum())

# Si hubiera NaNs, los quitamos (no debería ser mucho)
df_feat = df_feat.dropna().reset_index(drop=True)

print("\nShape después de dropna:", df_feat.shape)
print(df_feat["label"].value_counts())
df_feat.head()


In [ ]:
# Diferencia entre ojo izquierdo y derecho (parpadeo / asimetría)
df_feat["eye_open_diff"] = df_feat["eye_open_left"] - df_feat["eye_open_right"]

# Diferencia entre EAR de cada ojo
df_feat["ear_diff"] = df_feat["ear_left"] - df_feat["ear_right"]

# Relación extra de la boca (por si quieres)
df_feat["mouth_ratio_vh"] = df_feat["mouth_vertical"] / (df_feat["mouth_horizontal"] + 1e-6)

# Magnitud total del movimiento de la cabeza (como "energía" de movimiento)
df_feat["head_motion_mag"] = np.sqrt(
    df_feat["pitch"]**2 + df_feat["yaw"]**2 + df_feat["roll"]**2
)

# Actualizamos lista de features para el EDA
extended_features = feature_cols + [
    "eye_open_diff", "ear_diff", "mouth_ratio_vh", "head_motion_mag"
]

print("Total de features para EDA:", len(extended_features))


In [ ]:
group_stats = df_feat.groupby("label")[extended_features].describe().T
group_stats


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Asegúrate de que df_feat tenga 'label' + tus features
features = [
    "ear_left", "ear_right",
    "mar", "mouth_vertical", "mouth_horizontal",
    "brow_distance",
    "eye_open_left", "eye_open_right",
    "pitch", "yaw", "roll"
]

# Por si alguna no existe por algo:
features = [f for f in features if f in df_feat.columns]

# Definimos el grid (por ejemplo 4 filas x 3 columnas = 12 slots)
rows, cols = 4, 3

fig, axes = plt.subplots(rows, cols, figsize=(15, 15))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    sns.boxplot(data=df_feat, x="label", y=col, ax=ax)
    ax.set_title(f"{col} per label")
    ax.set_xlabel("")  # quitar texto extra abajo para que se vea más limpio
    ax.set_ylabel(col)

# Si sobran ejes (más subplots que features), los apagamos
for j in range(len(features), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Boxplots of features by user (real vs fake1 vs fake2)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
if "pitch" in df_feat.columns and "yaw" in df_feat.columns:
    plt.figure(figsize=(6,6))
    for label, sub in df_feat.groupby("label"):
        plt.scatter(sub["pitch"], sub["yaw"], alpha=0.5, label=label)
    plt.xlabel("Pitch (°)")
    plt.ylabel("Yaw (°)")
    plt.title("Head rotation: pitch vs yaw")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
if "mar" in df_feat.columns and "brow_distance" in df_feat.columns:
    plt.figure(figsize=(6,6))
    for label, sub in df_feat.groupby("label"):
        plt.scatter(sub["mar"], sub["brow_distance"], alpha=0.5, label=label)
    plt.xlabel("MAR (mouth aspect ratio)")
    plt.ylabel("Brow distance")
    plt.title("Mouth vs brows (real vs fake1 vs fake2)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
if "eye_open_diff" in df_feat.columns and "head_motion_mag" in df_feat.columns:
    plt.figure(figsize=(6,6))
    for label, sub in df_feat.groupby("label"):
        plt.scatter(sub["eye_open_diff"], sub["head_motion_mag"], alpha=0.5, label=label)
    plt.xlabel("Eye open diff (left - right)")
    plt.ylabel("Head motion magnitude")
    plt.title("Eye asymmetry vs Head motion")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
df_sorted = df_feat.sort_values(["label", "filename"]).reset_index(drop=True)

for angle in ["pitch", "yaw", "roll"]:
    if angle in df_feat.columns:
        plt.figure(figsize=(8,4))
        for label, sub in df_sorted.groupby("label"):
            plt.plot(sub[angle].values, marker=".", linestyle="-", alpha=0.6, label=label)
        plt.xlabel("Frame index (ordenado por filename)")
        plt.ylabel(f"{angle} (°)")
        plt.title(f"{angle} across the frames")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


In [ ]:
corr = df_feat[extended_features].corr()

plt.figure(figsize=(8,6))
im = plt.imshow(corr, interpolation='nearest')
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(extended_features)), extended_features, rotation=90)
plt.yticks(range(len(extended_features)), extended_features)
plt.title("Correlation matrix (through the samples)")
plt.tight_layout()
plt.show()


In [ ]:
for label, sub in df_feat.groupby("label"):
    corr_label = sub[extended_features].corr()
    plt.figure(figsize=(8,6))
    im = plt.imshow(corr_label, interpolation='nearest')
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(len(extended_features)), extended_features, rotation=90)
    plt.yticks(range(len(extended_features)), extended_features)
    plt.title(f"Correlation matrix - {label}")
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = df_feat[extended_features].values
y = df_feat["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

model = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(
        n_estimators=300,
        random_state=42
    )
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== Classification of real vs fake1 vs fake2 ===")
print(classification_report(y_test, y_pred))


In [ ]:
base_features = [
    "ear_left", "ear_right",
    "mar", "mouth_vertical", "mouth_horizontal",
    "brow_distance",
    "eye_open_left", "eye_open_right",
    "pitch", "yaw", "roll"
]

df_all["eye_open_diff"] = df_all["eye_open_left"] - df_all["eye_open_right"]
df_all["ear_diff"] = df_all["ear_left"] - df_all["ear_right"]
df_all["mouth_ratio_vh"] = df_all["mouth_vertical"] / (df_all["mouth_horizontal"] + 1e-6)
df_all["head_motion_mag"] = np.sqrt(
    df_all["pitch"]**2 + df_all["yaw"]**2 + df_all["roll"]**2
)

derived_features = ["eye_open_diff", "ear_diff", "mouth_ratio_vh", "head_motion_mag"]

feature_cols = base_features + derived_features


In [ ]:
from sklearn.model_selection import train_test_split

df_model = df_all[feature_cols + ["label"]].dropna().reset_index(drop=True)

X = df_model[feature_cols].values
y = df_model["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)



In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression
log_reg_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, multi_class="multinomial")
)
log_reg_model.fit(X_train, y_train)

# SVM
svm_model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", random_state=42)
)
svm_model.fit(X_train, y_train)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)
rf_model.fit(X_train, y_train)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

# 1. Get predictions from each model
y_pred_lr  = log_reg_model.predict(X_test)
y_pred_svm = svm_model.predict(X_test)
y_pred_rf  = rf_model.predict(X_test)

# 2. Compute metrics (accuracy + macro F1)
models = ["Logistic Regression", "SVM (RBF)", "Random Forest"]

accs = [
    accuracy_score(y_test, y_pred_lr),
    accuracy_score(y_test, y_pred_svm),
    accuracy_score(y_test, y_pred_rf),
]

f1s_macro = [
    f1_score(y_test, y_pred_lr, average="macro"),
    f1_score(y_test, y_pred_svm, average="macro"),
    f1_score(y_test, y_pred_rf, average="macro"),
]

print("Accuracies:", dict(zip(models, accs)))
print("Macro F1:", dict(zip(models, f1s_macro)))

# 3. Plot – bar chart comparing accuracy
x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(8,5))
plt.bar(x - width/2, accs, width, label="Accuracy")
plt.bar(x + width/2, f1s_macro, width, label="Macro F1")

plt.xticks(x, models, rotation=15)
plt.ylim(0, 1.0)
plt.ylabel("Score")
plt.title("Model Comparison – Accuracy and Macro F1")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred_rf = rf_model.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))

acc_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest accuracy:", acc_rf)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Make sure you already have:
# y_test  (true labels)
# y_pred_rf (predictions from Random Forest)

labels = ["real", "fake1", "fake2"]

cm = confusion_matrix(y_test, y_pred_rf, labels=labels)

plt.figure(figsize=(6,4))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=labels,
            yticklabels=labels)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Random Forest – Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Make sure you already have:
# y_test  (true labels)
# y_pred_rf (predictions from Random Forest)

labels = ["real", "fake1", "fake2"]

cm = confusion_matrix(y_test, y_pred_rf, labels=labels)

plt.figure(figsize=(6,4))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=labels,
            yticklabels=labels)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Random Forest – Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Feature importance desde el modelo RF
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(7,5))
plt.barh(
    np.array(feature_cols)[indices][:10],
    importances[indices][:10],
    color="teal"
)
plt.gca().invert_yaxis()
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
